# download packages

In [6]:
!pip install nltk


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: C:\Users\lucia\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
#NLTK imports
import nltk
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.corpus import stopwords
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('omw-1.4')

#other imports
import pandas as pd
import numpy as np
import ast
import re
import statistics

#statistics
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr, pearsonr

ModuleNotFoundError: No module named 'nltk'

In [ ]:
from nltk.stem import WordNetLemmatizer as wnl
from nltk.corpus import wordnet as wn
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')

from nltk.corpus import wordnet_ic
nltk.download('wordnet_ic')
brown_ic = wordnet_ic.ic('ic-brown.dat')


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet_ic to /root/nltk_data...
[nltk_data]   Unzipping corpora/wordnet_ic.zip.


In [ ]:
#load data (words and definitions) from which metrics for BM need to be extracted
df = pd.read_csv('data/EC.csv')

In [ ]:
#load psycholinguistic norm datasets
concreteness_df = pd.read_excel('data/psycholinguistic norms/Brisbaert_40K_Concreteness.xlsx')
physicality_df = pd.read_excel('data/psycholinguistic norms/SER_EN_5K.xls')
glasgow_df = pd.read_csv('data/psycholinguistic norms/glasgow_norms.csv')
mrc_c = pd.read_csv('data/psycholinguistic norms/MRC_corpus.csv')

glasgow_df = glasgow_df[1:]
keep_columns = ['Words', 'AROU', 'VAL', 'DOM', 'IMAG', 'FAM']
glasgow_df = glasgow_df[keep_columns]
glasgow_df = glasgow_df.astype({'AROU': float, 'VAL': float, 'DOM': float, 'IMAG': float, 'FAM': float})



In [ ]:
#download static models (this takes some time)
import gensim.downloader as api
nb_17 = api.load("conceptnet-numberbatch-17-06-300")
nb17_vocab = list(nb_17.key_to_index.keys())
nb17_vocab_en = [i.split('/')[3] for i in nb17_vocab if i.startswith('/c/en/')]

w2v = api.load('word2vec-google-news-300')
w2v_vocab = list(w2v.key_to_index.keys())

[==================================================] 100.0% 1168.7/1168.7MB downloaded
[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
#execute if only words and no definitions are provided
df['synsets'] = df['Word'].apply(lambda x: wordnet.synsets(x))
df = df.explode('synsets')
df.reset_index(drop=True, inplace=True)

AttributeError: 'float' object has no attribute 'lower'

In [ ]:
for i, row in df.iterrows():
  try:
    df.at[i, 'definitions'] = row['synsets'].definition()
  except:
    df.at[i, 'definitions'] = None
  try:
    df.at[i, 'examples'] = list(row['synsets'].examples())
  except:
    df.at[i, 'examples'] = None
  try:
    df.at[i, 'synonyms'] = [str(lemma.name()) for lemma in row['synsets'].lemmas()]
  except:
    df.at[i, 'synonyms'] = None

In [ ]:
df_vocab = []
for i, row in df.iterrows():
  df_vocab.append(row['Word'])
  try:
    for w in word_tokenize(row['definitions']):
      df_vocab.append(w)
  except: pass
print(len(df_vocab))
df_vocab = set(df_vocab)
print(len(df_vocab))
df_vocab = list(df_vocab)

8903
2143


#extend psycholinguistic norms

In [ ]:
#extend psycholinguistic features to cover oov
psycholinguistic_norms = pd.DataFrame(columns=['word', 'concreteness', 'physicality', 'imageability', 'familiarity', 'dataset'])
psycholinguistic_norms['word'] = df_vocab
for i, row in psycholinguistic_norms.iterrows():
  if row['word'] in concreteness_df['Word'].values:
    psycholinguistic_norms.at[i, 'dataset'] = 'concreteness'
    psycholinguistic_norms.at[i, 'concreteness'] = concreteness_df.loc[concreteness_df['Word'] == row['word'], 'Conc.M'].values[0]
  else:
    psycholinguistic_norms.at[i, 'concreteness'] = None

  if row['word'] in physicality_df['Word'].values:
    psycholinguistic_norms.at[i, 'dataset'] = 'physicality'
    psycholinguistic_norms.at[i, 'physicality'] = physicality_df.loc[physicality_df['Word'] == row['word'], 'Average SER'].values[0]
  else:
    psycholinguistic_norms.at[i, 'physicality'] = None

  if row['word'] in glasgow_df['Words'].values:
    psycholinguistic_norms.at[i, 'dataset'] = 'glasgow'
    psycholinguistic_norms.at[i, 'imageability'] = glasgow_df.loc[glasgow_df['Words'] == row['word'], 'IMAG'].values[0]
  else:
    psycholinguistic_norms.at[i, 'imageability'] = None

  if row['word'] in mrc_c[' word'].values:
    psycholinguistic_norms.at[i, 'dataset'] = 'mrc_c'
    psycholinguistic_norms.at[i, 'familiarity'] = mrc_c.loc[mrc_c[' word'] == row['word'], 'mrc.fam'].values[0]
  else:
    psycholinguistic_norms.at[i, 'familiarity'] = None

In [ ]:
dataset = physicality_df

nb17_gensim = []
for i, row in dataset.iterrows():
  try:
    nb17_gensim.append(nb_17['/c/en/'+ row['Word']])
  except:
    nb17_gensim.append(None)

dataset['nb17'] = nb17_gensim

w2v_glove = []
for i, row in dataset.iterrows():
  try:
    w2v_glove.append(w2v[row['Word']])
  except:
    w2v_glove.append(None)

dataset['w2v_glove'] = w2v_glove

In [ ]:
dataset=psycholinguistic_norms
nb17_gensim = []
for i, row in dataset.iterrows():
  try:
    nb17_gensim.append(nb_17['/c/en/'+ row['word']])
  except:
    nb17_gensim.append(None)
print(nb17_gensim)

dataset['nb17'] = nb17_gensim

w2v_glove = []
for i, row in dataset.iterrows():
  try:
    w2v_glove.append(w2v[row['word']])
  except:
    w2v_glove.append(None)

dataset['w2v_glove'] = w2v_glove

[array([ 0.1153, -0.0045,  0.0382, -0.1849, -0.0703, -0.0353,  0.0885,
        0.0454, -0.051 ,  0.1137, -0.0801, -0.0003,  0.0785, -0.0297,
        0.0661, -0.0324,  0.0047, -0.09  ,  0.0759, -0.051 ,  0.0137,
        0.0373,  0.0157, -0.0233, -0.093 ,  0.0559,  0.0204, -0.0269,
        0.0172, -0.0701,  0.1424,  0.1289,  0.0388, -0.0774,  0.0442,
        0.0471, -0.0823, -0.1161,  0.0849, -0.1171,  0.0163,  0.1432,
       -0.0358, -0.045 ,  0.0147,  0.0034, -0.0653,  0.0856,  0.0436,
        0.0874,  0.0508,  0.0285, -0.0787, -0.0151, -0.0439,  0.1103,
        0.0012, -0.0606, -0.0424, -0.0237,  0.1196, -0.0175,  0.0634,
       -0.0178,  0.0007,  0.1465, -0.0762,  0.0875,  0.0199, -0.0429,
        0.0125,  0.0028,  0.1239,  0.0535,  0.0844, -0.0361,  0.0423,
       -0.0807,  0.0055, -0.0082,  0.0379,  0.173 , -0.1051,  0.0389,
        0.0117, -0.0205, -0.007 ,  0.045 ,  0.106 ,  0.128 ,  0.0153,
        0.012 , -0.0203,  0.0762,  0.1681, -0.0114, -0.0299,  0.0383,
       -0.0591,  0.

In [ ]:
#show distributional features oov
w2v_oov = []
for word in df_vocab:
  if word not in w2v_vocab:
    w2v_oov.append(word)
print(len(w2v_oov))

nb_oov = []
for word in df_vocab:
  if word not in nb17_vocab:
    nb_oov.append(word)
print(len(nb_oov))

44
85


In [ ]:
train = physicality_df
test = psycholinguistic_norms[psycholinguistic_norms['physicality'].isna()]
print(len(train))
print(len(test))

max_len = max(len(x) if x is not None else 0 for x in train.nb17)
x_train = np.array([x if x is not None else np.zeros(max_len) for x in train.nb17])
y_train = train['Average SER'].values
max_len = max(len(x) if x is not None else 0 for x in test.nb17)
x_test = np.array([x if x is not None else np.zeros(max_len) for x in test.nb17])
print(x_train)

model = SVR(kernel='rbf', C=100, gamma=0.003, epsilon=.1)
model.fit(x_train, y_train)
preds = model.predict(x_test)

new_physicality = np.append(y_train, preds)
print(len(new_physicality))
psycholinguistic_norms['physicality'].update(new_physicality)

5857
1317
[[-0.0814      0.099       0.0155     ...  0.0904     -0.13600001
  -0.0154    ]
 [-0.0947      0.0641     -0.0259     ...  0.0036     -0.0939
  -0.0485    ]
 [ 0.0725      0.0592     -0.0272     ... -0.         -0.0068
  -0.0028    ]
 ...
 [ 0.0517     -0.0011      0.0419     ...  0.0442     -0.0062
  -0.024     ]
 [ 0.0047     -0.0101      0.092      ... -0.068      -0.0379
   0.0731    ]
 [ 0.099       0.0174      0.0578     ...  0.0086     -0.0405
   0.053     ]]
7174


<ipython-input-24-7b61daa925ed>:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  psycholinguistic_norms['physicality'].update(new_physicality)


In [ ]:
dataset = glasgow_df

nb17_gensim = []
for i, row in dataset.iterrows():
  try:
    nb17_gensim.append(nb_17['/c/en/'+ row['Words']])
  except:
    nb17_gensim.append(None)

dataset['nb17'] = nb17_gensim

w2v_glove = []
for i, row in dataset.iterrows():
  try:
    w2v_glove.append(w2v[row['Words']])
  except:
    w2v_glove.append(None)

dataset['w2v_glove'] = w2v_glove


In [ ]:
train = glasgow_df
test = psycholinguistic_norms[psycholinguistic_norms['imageability'].isna()]
print(len(train))
print(len(test))

max_len = max(len(x) if x is not None else 0 for x in train.nb17)
x_train = np.array([x if x is not None else np.zeros(max_len) for x in train.nb17])
y_train = train['IMAG'].values
x_test = np.array([x if x is not None else np.zeros(max_len) for x in test.nb17])

model = SVR(kernel='rbf', C=100, gamma=0.003, epsilon=.1)
model.fit(x_train, y_train)
preds = model.predict(x_test)

new_df = np.append(y_train, preds)
print(len(new_df))
psycholinguistic_norms['imageability'].update(new_df)

5553
1385
6938


<ipython-input-26-04559a18fb84>:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  psycholinguistic_norms['imageability'].update(new_df)


In [ ]:
dataset = mrc_c

nb17_gensim = []
for i, row in dataset.iterrows():
  try:
    nb17_gensim.append(nb_17['/c/en/'+ row[' word']])
  except:
    nb17_gensim.append(None)

dataset['nb17'] = nb17_gensim

w2v_glove = []
for i, row in dataset.iterrows():
  try:
    w2v_glove.append(w2v[row[' word']])
  except:
    w2v_glove.append(None)

dataset['w2v_glove'] = w2v_glove

In [ ]:
train = mrc_c
train = train.dropna(subset=['mrc.fam'])
test = psycholinguistic_norms[psycholinguistic_norms['familiarity'].isna()]

max_len = max(len(x) if x is not None else 0 for x in train.w2v_glove)
x_train = np.array([x if x is not None else np.zeros(max_len) for x in train.w2v_glove])
y_train = train['mrc.fam'].values

x_test = np.array([x if x is not None else np.zeros(max_len) for x in test.w2v_glove])

model = SVR(kernel='rbf', C=100, gamma=0.003, epsilon=.1)
model.fit(x_train, y_train)
preds = model.predict(x_test)

new_df = np.append(y_train, preds)
print(len(new_df))
psycholinguistic_norms['familiarity'].update(new_df)

10698


<ipython-input-29-f480c2510ab6>:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  psycholinguistic_norms['familiarity'].update(new_df)


In [ ]:
psycholinguistic_norms[psycholinguistic_norms['concreteness'].isna()]

,word,concreteness,physicality,imageability,familiarity,dataset,nb17,w2v_glove
0,levels,None,4.0,4.391,629.0,mrc_c,"[0.1153, -0.0045, 0.0382, -0.1849, -0.0703, -0...","[-0.16113281, 0.09472656, 0.15527344, 0.296875..."
3,distributors,None,1.727273,2.516,566.0,mrc_c,"[-0.0031, -0.0357, 0.0107, 0.0051, -0.1192, -0...","[-0.028808594, -0.041015625, -0.04663086, 0.26..."
4,having,None,2.272727,2.571,403.0,mrc_c,"[0.1725, 0.1359, 0.0307, -0.0305, 0.0028, -0.0...","[-0.053955078, 0.05883789, -0.12792969, 0.1245..."
10,efforts,None,4.5,2.84,524.0,mrc_c,"[0.1242, 0.0823, -0.0505, -0.0699, -0.013, -0....","[0.11328125, 0.30664062, -0.10644531, 0.143554..."
14,Job,None,1.909091,4.452,552.0,NaN,None,"[0.20605469, -0.053466797, -0.31835938, -0.265..."
...,...,...,...,...,...,...,...,...
2132,commands,None,4.0,6.194,504.0,mrc_c,"[0.1034, 0.096, -0.0683, -0.1117, -0.054, 0.11...","[0.3203125, 0.111328125, -0.021240234, 0.01226..."
2134,takes,None,2.181818,2.057,526.0,mrc_c,"[0.1648, 0.1312, -0.0016, -0.0455, -0.0783, -0...","[0.12451172, 0.09082031, -0.0048828125, -0.152..."
2137,shocks,None,3.727273,6.441,243.0,mrc_c,"[0.1471, 0.0356, 0.0938, -0.0773, 0.0, -0.0996...","[-0.24902344, 0.37304688, -0.009643555, 0.2148..."
2138,institutions,None,2.909091,3.75,147.0,mrc_c,"[0.0481, 0.0357, -0.0924, -0.2127, -0.0077, 0....","[-0.07763672, 0.13964844, 0.15820312, 0.433593..."


In [ ]:
#download psycholinguistic norms extended to cover the current df vocabulary
psycholinguistic_norms.to_csv('/content/drive/MyDrive/PhD/ONGOING PROJECTS/METAFORAS/Anotaciones/Detect BM/psycholinguistic_data/psycholinguistic_norms_VUA_train.tsv', sep='\t')

# Add distributional features to df

In [ ]:
wnl = nltk.WordNetLemmatizer()
def process_definitions (x):
  sentence = word_tokenize(x)
  sentence = [word for word in sentence if word not in stopwords.words('english')]
  sentence = [wnl.lemmatize(word) for word in sentence]
  return sentence

In [ ]:
df['procesed_definitions'] = df['definitions'].apply(process_definitions) #3min

In [ ]:
nb17_vocab_set = set(nb17_vocab_en)  # Convert nb17_vocab to a set for faster lookups
df['processed_definitions_1'] = df['procesed_definitions'].apply(lambda x: [w for w in x if w in nb17_vocab_set])

In [ ]:
def nb17_cosines_definitions (sentence, target_word):
    sentence = [word for word in sentence if word != target_word]
    if target_word in nb17_vocab_set:
      try:
        return [nb_17.similarity('/c/en/'+ target_word, '/c/en/'+ word) for word in sentence]
      except:
        print(sentence)
    else:
      return None

In [ ]:
def w2v_cosines_definitions (sentence, target_word):
  sentence = [word for word in sentence if word != target_word]
  if target_word in w2v_vocab:
    return [w2v.similarity(target_word, word) for word in sentence]
  else:
    return None

In [ ]:
df['nb17_cosines_definitions'] = df.apply(lambda row: nb17_cosines_definitions(row['processed_definitions_1'], row['word']), axis=1)

In [ ]:
w2v_vocab_set = set(w2v_vocab)
df['processed_definitions_1'] = df['procesed_definitions'].apply(lambda x: [w for w in x if w in w2v_vocab_set])
df['w2v_cosines_definitions'] = df.apply(lambda row: w2v_cosines_definitions(row['processed_definitions_1'], row['word']), axis=1)

#Add psycholinguistic features to df

In [ ]:
psycholinguistic_norms = pd.read_csv('/content/drive/MyDrive/PhD/ONGOING PROJECTS/METAFORAS/Anotaciones/Detect BM/psycholinguistic_data/psycholinguistic_norms_VUA_train.tsv', sep='\t')

In [ ]:
psycholinguistic_norms_dict = {}
for i, row in psycholinguistic_norms.iterrows():
  psycholinguistic_norms_dict[row['word']] = row

In [ ]:
def check_norm (sentence, feature):
  sent_norms = []
  for word in sentence:
    if word in psycholinguistic_norms_dict:
      sent_norms.append(psycholinguistic_norms_dict[word][feature])
    else:
      sent_norms.append(None)
  return sent_norms

In [ ]:
#compute psycholinguistic features
for feature in ['familiarity', 'imageability', 'concreteness', 'physicality']:
  definitions = []
  for i, row in df.iterrows():
    definitions.append(check_norm(row['procesed_definitions'], feature))
  print(f'feature {feature} done ')
  df[f'{feature}_definitions'] = definitions

#Add precission measures to df

In [ ]:
def compute_precission (sentence, target_word): #tokenized_sentence
  def_synsets = []
  def_precission = []
  def_ic = []
  for w in sentence:
    if wordnet.synsets(w) and target_word != w:
      try:
        lesk_syn = lesk(sentence, w, synsets=wn.synsets(w))
        lesk_syn_tw = lesk(sentence, target_word, synsets=wn.synsets(target_word))
        def_synsets.append(lesk_syn)
        def_precission.append(lesk_syn.min_depth())
      except:
        def_synsets.append(None)
        def_precission.append(None)
      try:
        def_ic.append(lesk_syn.res_similarity(lesk_syn_tw, brown_ic))
      except:
        def_ic.append(None)
    else:
      def_synsets.append(None)
      def_precission.append(None)
      def_ic.append(None)
  return (def_precission, def_ic)

In [ ]:
#compute precission
definition_precission = []
definition_p_ic = []

for i, row in df.iterrows():
  definition_precission.append(compute_precission(word_tokenize(row['definitions']), row['word'])[0])
  definition_p_ic.append(compute_precission(word_tokenize(row['definitions']), row['word'])[1])

df['precission_definitions'] = definition_precission
df['ic_precission_definitions'] = definition_p_ic


#Max min mean

In [ ]:
column_list = []
for column in df.columns:
  if column.endswith('_definitions'):
    column_list.append(column)

In [ ]:
column_list

In [ ]:
df['nb17_cosines_definitions'] = df['nb17_cosines_definitions'].apply(lambda x: [float(n) for n in x if n is not None] if x is not None else [])
df['w2v_cosines_definitions'] = df['w2v_cosines_definitions'].apply(lambda x: [float(n) for n in x if n is not None] if x is not None else [])

In [ ]:
def mean(x):
  x = [l for l in (x) if type(l) == int or type(l) == float or type(l) == np.float64 or type(l) == np.float32]
  if len(x) == 0:
    return None
  else:
    return sum(x)/len(x)

In [ ]:
for column in column_list:
  df[f'{column}_mean'] =  df[column].apply(mean)

In [ ]:
for column in column_list:
  print(column)
  max_vals = []
  min_vals = []
  for i, row in df.iterrows():
    a = [l for l in (row[column]) if type(l) == int or type(l) == float or type(l) == np.float64]
    if len(a) == 0:
      max_vals.append(None)
      min_vals.append(None)
    else:
      max_vals.append(max(a))
      min_vals.append(min(a))
  df[f'{column}_max'] = max_vals
  df[f'{column}_min'] = min_vals

#Select best BM definitions and download

In [ ]:
#Select best BM: first group all definitions per word
word_index = []
counter = 0
last_word = ''
for i, row in df.iterrows():
  if row['word'] != last_word:
    last_word = row['word']
    counter+=1
    word_index.append(counter)
  else:
    word_index.append(counter)
df['word_index'] = word_index

In [ ]:
column_list = ['nb17_cosines_definitions_mean',
 'w2v_cosines_definitions_mean',
 'familiarity_definitions_mean',
 'imageability_definitions_mean',
 'concreteness_definitions_mean',
 'physicality_definitions_mean',
 'precission_definitions_mean',
 'ic_precission_definitions_mean']

In [ ]:
df['best_bm'] = 0

In [ ]:
counter

In [ ]:
import statistics

In [ ]:
for i in range (counter)[1:]:
  sup_df = df[df['word_index']==i]
  best_rows = []
  for column in column_list:
    best_rows.append(sup_df[column].idxmax())
  if best_rows:
    try:
      df.loc[statistics.mode(best_rows), 'best_bm'] = 1
    except:
      pass

In [ ]:
df.groupby('best_bm').size()

In [ ]:
df

In [ ]:
df.to_csv('/content/drive/MyDrive/VUA_test_BM_1.tsv', sep='\t', index=False)